#### ENRICH `PRODUCT_CATEGORY` FROM BRONZE TO SILVER

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'PRODUCT_CATEGORY_ENRICHED'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
SOURCE = f"{CATALOG}.BRONZE.PRODUCT_CATEGORY"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
product_category_df = spark.table(SOURCE)
product_category_df.limit(1).display()

In [0]:
from pyspark.sql.functions import current_timestamp

product_category_trans_df = product_category_df.select(
    "product_category_name",
    "product_category_name_english",
    "file_path",
    current_timestamp().alias("load_timestamp")
).withColumnRenamed("file_path", "origin_file_path")

product_category_trans_df.limit(1).display()

#### FULL LOAD `ORDERS_SALES.SILVER.PRODUCT_CATEGORY_ENRICHED`

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def write_audit(run_id, status, start_time, rows=None, error=None):
    end = datetime.now()
    row = (run_id, "PRODUCT_CATEGORY_ENRICHED_LOAD", "SILVER", SOURCE, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
    (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
    (product_category_trans_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET))

    # rows written by this overwrite, from the Delta commit metrics (no extra scan)
    metrics = (DeltaTable.forName(spark, TARGET)
                 .history(1).select("operationMetrics").first()[0])
    rows_written = int(metrics.get("numOutputRows", 0))

except Exception:
    status, err = "FAILED", traceback.format_exc()
    raise
finally:
    write_audit(run_id, status, start_time, rows_written, err)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.PRODUCT_CATEGORY_ENRICHED')